<a href="https://colab.research.google.com/github/lricci03/challengedata/blob/main/joint_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Joint model: RNN and engineered features

In [1]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Sequential Features and Target

In [13]:
df = pd.read_parquet("/content/drive/MyDrive/Challengedata/Xtrain.parquet")

In [14]:
df_num = df[['obs_id','price', 'bid', 'ask', 'bid_size', 'ask_size', 'flux', 'action', 'venue']]
df_num['venue'] = df_num['venue'].apply(str)
df_num = pd.get_dummies(df_num, dtype=float)

/tmp/ipykernel_1068/4138682116.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_num['venue'] = df_num['venue'].apply(str)


Don't need df anymore

In [15]:
import gc

del df
gc.collect()

0

In [16]:
targets = pd.read_csv("/content/drive/MyDrive/Challengedata/y_train_or6m3Ta.csv")

Ensure alignment

In [17]:
targets = targets.set_index("obs_id")
df_num = df_num.set_index("obs_id")

Reshape X. This reinterprets every consecutive block of 100 rows as one sample. The -1 means: infer this dimension automatically.

In [18]:
X_seq = torch.tensor(df_num.values, dtype=torch.float32)
X_seq = X_seq.reshape(-1, 100, 15)
y = torch.tensor(targets.values.squeeze(), dtype=torch.long)

Delete the dataframes

In [19]:
del df_num, targets
gc.collect()

0

## Engineered Features

In [20]:
df_eng = pd.read_csv("/content/drive/MyDrive/Challengedata/eng_features.csv")

In [23]:
df_eng.head()

,obs_id,median_price,median_mid_price,mean_spread,median_spread,std_spread,min_spread,max_spread,mean_bid_size,median_bid_size,...,std_flux,add_ratio,delete_ratio,update_ratio,venue_4_ratio,venue_1_ratio,venue_5_ratio,venue_2_ratio,venue_0_ratio,venue_3_ratio
0,0,0.00,0.010,0.0152,0.02,0.005021,0.01,0.02,190.60,100.0,...,93.739861,0.51,0.49,0.00,0.59,0.14,0.07,0.13,0.05,0.02
1,1,0.01,0.040,0.0392,0.04,0.006917,0.02,0.06,290.94,300.0,...,94.011927,0.49,0.43,0.08,0.51,0.07,0.21,0.10,0.06,0.05
2,2,0.01,0.005,0.0107,0.01,0.002564,0.01,0.02,1902.15,1510.0,...,151.864373,0.54,0.46,0.00,0.44,0.02,0.19,0.01,0.11,0.23
3,3,0.21,0.150,0.1310,0.10,0.046482,0.10,0.20,275.13,200.0,...,74.986447,0.50,0.49,0.01,0.43,0.18,0.09,0.11,0.18,0.01
4,4,0.02,0.020,0.0174,0.02,0.004408,0.01,0.02,143.15,122.5,...,126.849955,0.48,0.52,0.00,0.42,0.09,0.18,0.02,0.19,0.10


In [24]:
df_eng = df_eng.set_index("obs_id")

In [25]:
X_eng = torch.tensor(df_eng.values, dtype=torch.float32)

In [26]:
X_seq.shape, X_eng.shape, y.shape

(torch.Size([160800, 100, 15]), torch.Size([160800, 30]), torch.Size([160800]))

## Train and Validation set

In [27]:
X_seq_train = X_seq[:128640]
X_seq_valid = X_seq[128640:]

X_eng_train = X_eng[:128640]
X_eng_valid = X_eng[:128640:]

y_train = y[:128640]
y_valid = y[128640:]

## Scaling

In [28]:
torch.isnan(X_eng).sum()

tensor(0)

Compute mean and standard deviation of the test set, then apply the scaling to both train and validation.

In [30]:
mean = X_eng_train.mean(dim=0, keepdim=True)
std = X_eng_train.std(dim=0, keepdim=True)

# avoid division by zero for constant features
std[std == 0] = 1.0

X_eng_train = (X_eng_train - mean) / std
X_eng_valid = (X_eng_valid - mean) / std

## Dataset and DataLoader

In [31]:
from torch.utils.data import Dataset
import torch

class StockDataset(Dataset):
    def __init__(self, X_seq, X_eng, y):
        self.X_seq = X_seq
        self.X_eng = X_eng
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_seq[idx],
            self.X_eng[idx],
            self.y[idx]
        )

In [32]:
train_dataset = StockDataset(X_seq_train, X_eng_train, y_train)
valid_dataset = StockDataset(X_seq_valid, X_eng_valid, y_valid)

In [33]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
valid_loader = DataLoader(valid_dataset, batch_size=64)

## Joint GRU model

In [34]:
import torch
import torch.nn as nn

class JointGRUModel(nn.Module):
    def __init__(self, n_seq_features, n_engineered_features, hidden_size=128, n_classes=24):
        super().__init__()

        self.gru = nn.GRU(
            input_size=n_seq_features,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.classifier = nn.Linear(
            hidden_size + n_engineered_features,
            n_classes
        )

    def forward(self, x_seq, x_eng):

        # x_seq: (batch_size, 100, n_seq_features)

        outputs, last_state = self.gru(x_seq)

        # For a 1-layer GRU:
        # outputs shape: [batch_size, window length (100), hidden_size)
        # outputs[:, -1] shape: [batch_size, hidden_size]
        h_gru = outputs[:, -1]

        # x_eng:
        # (batch_size, n_engineered_features)

        x = torch.cat([h_gru, x_eng], dim=1)

        # x:
        # (batch_size, 128 + n_engineered_features)

        logits = self.classifier(x)

        return logits

## Train and evaluation functions

We need to modify slightly since the batches are now of the form: x_seq, x_eng, y instead of previously: x y

In [36]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 15.3 MB/s eta 0:00:00


In [42]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_seq, X_eng, y in data_loader:
            X_seq, X_eng, y = X_seq.to(device), X_eng.to(device), y.to(device)

            logits = model(X_seq, X_eng)

            y_class = logits.argmax(dim=1)
            metric.update(y_class, y)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=20, factor=0.1):

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max", # we monitor accuracy
        patience=patience,
        factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_seq, X_eng, y in train_loader:
            X_seq, X_eng, y = X_seq.to(device), X_eng.to(device), y.to(device)

            logits = model(X_seq, X_eng)

            loss = loss_fn(logits, y)

            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            y_class = logits.argmax(dim=1)
            metric.update(y_class, y)

        train_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()
        val_metric = evaluate_tm(model, valid_loader, metric).item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(val_metric)

        scheduler.step(val_metric)

        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

## Train the model

In [43]:
torch.manual_seed(42)

joint_model = JointGRUModel(n_seq_features=15, n_engineered_features=30, hidden_size=128, n_classes=24).to(device)

check the gpu before training

In [45]:
torch.cuda.is_available()

False

In [44]:
torch.manual_seed(42)

optimizer = torch.optim.NAdam(joint_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=24).to(device)

history = train(joint_model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=50)

KeyboardInterrupt: 